### 1. 数据与原函数

在这个连接中查看[数据与原函数](https://www.rethink.fun/chapter6/Normalization.html)，以表格中的数据为基础进行下面训练

### 2. 用梯度下降训练

In [2]:
import torch

# 指定设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 输入特征与对应标签
inputs = torch.tensor(
    [[2, 1000], [3, 2000], [2, 500], [1, 800], [4, 3000]],
    dtype=torch.float,
    device=device,
)
labels = torch.tensor([[19], [31], [14], [15], [43]], dtype=torch.float, device=device)


# 定义线性回归模型：
class MLR:
    def __init__(self, x):
        self.x = x
        self.w = torch.ones(2, 1, requires_grad=True, device=device)
        self.b = torch.ones(1, requires_grad=True, device=device)

    def output(self):
        return self.x @ self.w + self.b


# 定义损失函数
def Loss(output, target):
    differ = output - target
    square_num = torch.square(differ)
    mean_loss = torch.mean(square_num)
    return mean_loss


# 设置超参数
epoch_num = 200
lr = 0.0000001

# 开始训练
model = MLR(inputs)
for epoch in range(epoch_num):
    y_pred = model.output()
    loss = Loss(y_pred, labels)
    loss.backward()

    print(f"w_grad: {model.w.grad.tolist()}")
    with torch.no_grad():
        model.w -= lr * model.w.grad
        model.b -= lr * model.b.grad

    # 清空梯度
    model.w.grad.zero_()
    model.b.grad.zero_()

    print(f"epoch {epoch}: loss {loss.item():.5f}")

print(f"最终的权重为：\nw: {model.w}, b: {model.b}")

w_grad: [[8600.0], [5876040.0]]
epoch 0: loss 2898583.75000
w_grad: [[3476.080322265625], [2376262.5]]
epoch 1: loss 474035.15625
w_grad: [[1403.973876953125], [960956.9375]]
epoch 2: loss 77528.75000
w_grad: [[566.017333984375], [388609.625]]
epoch 3: loss 12684.82715
w_grad: [[227.14889526367188], [157153.21875]]
epoch 4: loss 2080.36768
w_grad: [[90.11094665527344], [63552.5390625]]
epoch 5: loss 346.13290
w_grad: [[34.6929817199707], [25700.552734375]]
epoch 6: loss 62.51920
w_grad: [[12.282051086425781], [10393.2734375]]
epoch 7: loss 16.13755
w_grad: [[3.2191028594970703], [4203.03076171875]]
epoch 8: loss 8.55237
w_grad: [[-0.44594287872314453], [1699.704345703125]]
epoch 9: loss 7.31190
w_grad: [[-1.9280805587768555], [687.363037109375]]
epoch 10: loss 7.10904
w_grad: [[-2.5274596214294434], [277.970947265625]]
epoch 11: loss 7.07586
w_grad: [[-2.7698497772216797], [112.4111328125]]
epoch 12: loss 7.07043
w_grad: [[-2.867867946624756], [45.4619140625]]
epoch 13: loss 7.06954
w_

### 3. 为什么要对feature进行归一化

通过上边的代码，可以训练一个线性回归模型，你可以尝试调整学习率lr，你会发现这个lr必须设置的很小。如果设置稍大，模型训练过程就会不收敛，loss值会快速增大，直到超过float的表示范围。而且loss值降到7左右，就很难再下降了。我们造的数据是严格按照线性方程构造的，理论上loss应该可以降到非常接近0的。但为什么loss值不能下降到0呢？ 我们仔细观察第一次迭代的打印值：

可以发现，对 `lights` 权重 $w_0$ 的梯度值约为 **8600**，对 `distance` 权重 $w_1$ 的梯度值约为 **5,876,040**。

- $w_1$ 的梯度大约是 $w_0$ 梯度的 **1000 倍**：
  $$\left|\frac{\partial L}{\partial w_1}\right| \approx 1000 \left|\frac{\partial L}{\partial w_0}\right|$$

原因是：

- $w_1$ 作用在特征 `distance` 上，而 $w_0$ 作用在特征 `lights` 上。
- `distance` 的数值量级大约是 `lights` 的 **1000 倍**。
- 在最终的损失函数 $L$ 中，当我们对参数做“同样幅度”的改变时，作用在更大尺度特征（`distance`）上的权重变化，会对输出与损失产生更大的影响（大约放大 1000 倍）。

因此，两个权重对应的梯度值会相差约 **1000 倍**。

---

初始化时，$w_0$ 和 $w_1$ 都是 1。最终我们希望：

- $w_0$ 调整到 2  
- $w_1$ 调整到 0.01

但我们看到 $w_1$ 的梯度值非常大。如果学习率稍微大一些，更新公式

$$w_1 \leftarrow w_1 - lr \cdot \frac{\partial L}{\partial w_1}$$

中的 $lr \cdot \frac{\partial L}{\partial w_1}$ 就会变得很大，导致从 1 一步“跨过”目标值 0.01（甚至直接变成负数或发散）。这就是为什么学习率必须设置得很小。

为了迁就对 $w_1$ 的稳定调整，学习率不得不设置得很小，但这会带来另一个问题：对 $w_0$ 来说，每次更新量

$$lr \cdot \frac{\partial L}{\partial w_0}$$

又会变得太小，从而导致训练非常慢。这也解释了为什么当 loss 降到 7 左右就很难再下降。

根本原因在于：$w_0$ 和 $w_1$ 的训练 **共用同一个学习率**，但它们对应的 feature 取值范围不同，导致它们对 loss 的影响程度不同，进一步导致它们梯度的数值范围也不同（一个梯度很大、一个梯度相对较小）。因此，用同一个学习率同时更新两个参数时，就很难兼顾二者的收敛速度与稳定性。

### 4. 进行归一化

In [ ]:
如果我们让所有 feature 的取值范围相同，那么所有训练参数对 loss 函数的影响就会更接近，计算得到的梯度也会比较“同量级”，从而可以用统一的学习率来进行调整。

对于 bias 而言，它的系数恒为 1，相当于它的输入 feature 大小永远都是 1。那么我们就把其他 feature 也调整到 1 左右。

最简单的做法就是：让输入 feature 都除以该 feature 的最大值，这样所有 feature 的取值都会落在 0 到 1 之间。

我们试一下这样是否可以改进训练的稳定性：

In [3]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = torch.tensor(
    [[2, 1000], [3, 2000], [2, 500], [1, 800], [4, 3000]],
    dtype=torch.float,
    device=device,
)
labels = torch.tensor([[19], [31], [14], [15], [43]], dtype=torch.float, device=device)

# 进行归一化
inputs = inputs / torch.tensor([4, 3000], device=device)


w = torch.ones(2, 1, requires_grad=True, device=device)
b = torch.ones(1, requires_grad=True, device=device)

epoch = 1000
lr = 0.5

for i in range(epoch):
    outputs = inputs @ w + b
    loss = torch.mean(torch.square(outputs - labels))
    print("loss", loss.item())
    loss.backward()
    print("w.grad", w.grad.tolist())
    with torch.no_grad():
        w -= w.grad * lr
        b -= b.grad * lr

    w.grad.zero_()
    b.grad.zero_()

loss 609.12255859375
w.grad [[-31.823333740234375], [-28.171554565429688]]
loss 276.1470947265625
w.grad [[18.713247299194336], [14.430887222290039]]
loss 130.7478485107422
w.grad [[-14.165596008300781], [-13.107967376708984]]
loss 66.27213287353516
w.grad [[7.567931175231934], [5.2582902908325195]]
loss 36.88682556152344
w.grad [[-6.486094951629639], [-6.472206115722656]]
loss 22.863407135009766
w.grad [[2.8819406032562256], [1.4812228679656982]]
loss 15.68340015411377
w.grad [[-3.105940341949463], [-3.4828484058380127]]
loss 11.647061347961426
w.grad [[0.949533224105835], [-0.009576082229614258]]
loss 9.129942893981934
w.grad [[-1.5856672525405884], [-2.083230972290039]]
loss 7.404510498046875
w.grad [[0.18418240547180176], [-0.5427928566932678]]
loss 6.133534908294678
w.grad [[-0.8760228753089905], [-1.3865240812301636]]
loss 5.151700973510742
w.grad [[-0.09213244915008545], [-0.6842262744903564]]
loss 4.371189117431641
w.grad [[-0.5246062278747559], [-1.0085372924804688]]
loss 3.74

### 5. 对特征进行标准化

实际上在深度学习里，更常用的是对特征进行**标准化**处理：对每个 feature 减去自己的均值，再除以自己的标准差。这样就把这个 feature 转化为**均值为 0、标准差为 1** 的分布。

在**归一化**操作里，是对每个 feature 除以这个 feature 在所有样本中的**绝对值最大值**。也就是说，只有这一个值决定缩放大小，但这个值有可能是个异常值。

相比之下，**标准化**会考虑所有样本的分布情况，避免缩放受异常值的影响，训练起来会更稳定。

In [7]:
# 定义normalize函数
def normlize(inputs):
    # 计算所有样本相同维度特征的均值与标准差
    mean = inputs.mean(dim=0)
    std = inputs.std(dim=0)
    # 特征标准化
    return (inputs - mean) / std

In [9]:
import torch

# 指定设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 输入特征与对应标签
inputs = torch.tensor(
    [[2, 1000], [3, 2000], [2, 500], [1, 800], [4, 3000]],
    dtype=torch.float,
    device=device,
)
labels = torch.tensor([[19], [31], [14], [15], [43]], dtype=torch.float, device=device)


# 定义线性回归模型：
class MLR:
    def __init__(self, x):
        self.x = x
        self.w = torch.ones(2, 1, requires_grad=True, device=device)
        self.b = torch.ones(1, requires_grad=True, device=device)

    def output(self):
        return self.x @ self.w + self.b


# 定义损失函数
def Loss(output, target):
    differ = output - target
    square_num = torch.square(differ)
    mean_loss = torch.mean(square_num)
    return mean_loss


# 设置超参数
epoch_num = 1000
lr = 0.5

# 特征标准化
inputs = normlize(inputs)

# 开始训练
model = MLR(inputs)
for epoch in range(epoch_num):
    y_pred = model.output()
    loss = Loss(y_pred, labels)
    loss.backward()

    print(f"w_grad: {model.w.grad.tolist()}")
    with torch.no_grad():
        model.w -= lr * model.w.grad
        model.b -= lr * model.b.grad

    # 清空梯度
    model.w.grad.zero_()
    model.b.grad.zero_()

    print(f"epoch {epoch}: loss {loss.item():.5f}")

print(f"最终的权重为：\nw: {model.w}, b: {model.b}")

w_grad: [[-15.604002952575684], [-16.726499557495117]]
epoch 0: loss 635.20972
w_grad: [[9.087748527526855], [8.043952941894531]]
epoch 1: loss 25.92263
w_grad: [[-4.0536723136901855], [-5.024291038513184]]
epoch 2: loss 8.41302
w_grad: [[2.8564586639404297], [1.9538872241973877]]
epoch 3: loss 3.34310
w_grad: [[-0.8548362255096436], [-1.6941308975219727]]
epoch 4: loss 1.78682
w_grad: [[1.0655667781829834], [0.2851123809814453]]
epoch 5: loss 1.23509
w_grad: [[0.005011647939682007], [-0.720727801322937]]
epoch 6: loss 0.98095
w_grad: [[0.5270583033561707], [-0.14780156314373016]]
epoch 7: loss 0.82379
w_grad: [[0.21328715980052948], [-0.4142605662345886]]
epoch 8: loss 0.70547
w_grad: [[0.3450268507003784], [-0.2385251522064209]]
epoch 9: loss 0.60809
w_grad: [[0.24310296773910522], [-0.2995377480983734]]
epoch 10: loss 0.52528
w_grad: [[0.26724934577941895], [-0.23734888434410095]]
epoch 11: loss 0.45406
w_grad: [[0.2266899198293686], [-0.24253252148628235]]
epoch 12: loss 0.39258
w_

### 6. 预测时的归一化

有一点要特别注意，假如你在训练时对数据做了归一化，那么你一定要记录你做归一化时的参数。在对数据进行预测时，首先需要先对feature用同样的参数进行归一化，然后再带入模型，得到预测值。

即：

- 如果你训练时做的是**标准化（standardization）**：用训练集计算出的 **mean（均值）** 和 **std（标准差）**，在预测时对新的输入也做同样的变换：  
  $$x'=\frac{x-\mu_{\text{train}}}{\sigma_{\text{train}}}$$

- 如果你训练时做的是**归一化（min-max 或除以最大值）**：就用训练集算出来的 **min/max** 或 **max_abs**，预测时同样用这些训练集参数去缩放新输入。

这样做的原因是：模型学到的参数 $w,b$ 是在“被你处理过的特征空间”里拟合出来的；如果预测时换了一套新的均值/标准差（用预测样本自己算），输入分布就变了，模型的 $w$ 对应的尺度也不匹配，预测会不稳定甚至明显偏差。

Min-Max 指的是一种最常见的“归一化（Normalization）”方法：把特征的取值线性缩放到固定区间（通常是 $[0,1]$）。

**定义（把 $x$ 缩放到 $[0,1]$）：**
- 对某个特征，先在训练集里计算  
  $$x_{\min}=\min(x),\quad x_{\max}=\max(x)$$
- 然后对每个样本做变换：  
  $$x'=\frac{x-x_{\min}}{x_{\max}-x_{\min}}$$

这样：
- 当 $x=x_{\min}$ 时，$x'=0$
- 当 $x=x_{\max}$ 时，$x'=1$

**举个小例子：** 假设某个特征在训练集里最小值是 500，最大值是 3000。新样本 $x=2500$：
$$x'=\frac{2500-500}{3000-500}=\frac{2000}{2500}=0.8$$

**和你前面“除以最大值”有什么关系？**
- 你前面做的 `x' = x / x_max` 也是一种归一化，但它不一定把最小值映射到 0（除非 $x_{\min}=0$）。
- Min-Max 更“标准”，能确保范围严格落在 $[0,1]$（训练集范围内）。

**预测时也要用训练集的 $x_{\min}, x_{\max}$**，不能用预测样本自己算。

In [12]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = torch.tensor(
    [[2, 1000], [3, 2000], [2, 500], [1, 800], [4, 3000]],
    dtype=torch.float,
    device=device,
)
labels = torch.tensor([[19], [31], [14], [15], [43]], dtype=torch.float, device=device)

# 计算每个特征的均值和标准差
mean = inputs.mean(dim=0)
std = inputs.std(dim=0)
# 对特征进行标准化
inputs = (inputs - mean) / std

w = torch.ones(2, 1, requires_grad=True, device=device)
b = torch.ones(1, requires_grad=True, device=device)

epoch = 2000
lr = 0.1

for i in range(epoch):
    outputs = inputs @ w + b
    loss = torch.mean(torch.square(outputs - labels))
    print("loss", loss.item())
    loss.backward()
    print("w.grad", w.grad.tolist())
    with torch.no_grad():
        w -= w.grad * lr
        b -= b.grad * lr

    w.grad.zero_()
    b.grad.zero_()

# 对新采集的数据进行预测
new_input = torch.tensor([[3, 2500]], dtype=torch.float, device=device)
# 对于新的数据进行预测时，同样要进行标准化
new_input = (new_input - mean) / std
# 预测
predict = new_input @ w + b
# 打印预测结果
print("Predict:", predict.tolist()[0][0])

loss 635.209716796875
w.grad [[-15.604002952575684], [-16.726499557495117]]
loss 393.75823974609375
w.grad [[-10.665654182434082], [-11.772409439086914]]
loss 246.21743774414062
w.grad [[-7.240629196166992], [-8.331866264343262]]
loss 155.147216796875
w.grad [[-4.865854263305664], [-5.9417901039123535]]
loss 98.46870422363281
w.grad [[-3.219943046569824], [-4.280791759490967]]
loss 62.9586296081543
w.grad [[-2.0798492431640625], [-3.1258225440979004]]
loss 40.59088897705078
w.grad [[-1.290771722793579], [-2.3220791816711426]]
loss 26.4392032623291
w.grad [[-0.7452747821807861], [-1.7621214389801025]]
loss 17.45209312438965
w.grad [[-0.36879873275756836], [-1.3713877201080322]]
loss 11.725461959838867
w.grad [[-0.10959887504577637], [-1.0981298685073853]]
loss 8.064143180847168
w.grad [[0.06824278831481934], [-0.9064277410507202]]
loss 5.714571952819824
w.grad [[0.18964266777038574], [-0.7713612914085388]]
loss 4.19988489151001
w.grad [[0.2719001770019531], [-0.6756290197372437]]
loss 3

### 7. 为什么归一化不会影响模型？

你可能好奇：归一化明明改变了数据，为什么不会影响训练出来的模型呢？

这是因为归一化本质上只是对数据做了**可逆的线性变换**（在参数空间里相当于“换了一个坐标系”），模型的**理论表达能力并没有变**，也不会改变数据之间的本质关系。这一现象类似于“**换单位不会影响物理规律**”。

因此，归一化通常不会影响深度学习模型最终能达到的训练结果：它只是对数据做线性缩放，保留了所有必要的信息；模型可以通过调整权重来**完全补偿**这种变换。

把“在参数空间里相当于换了一个坐标系”理解成一句话就是：

你把输入特征的尺度改了（比如把 `distance` 从“米”改成“千米”），那么同一个模型在参数里的“最佳权重数值”也会跟着按比例变化；本质上模型表达的是同一条规律，只是参数的数值坐标换了一套刻度。

**用最简单的线性回归举例（1 个特征）：**  
原模型：
$$y = wx + b$$

如果你把特征做缩放（归一化/标准化里的一部分），比如
$$x' = \frac{x}{s}\quad (s>0)$$

那同一个函数关系也可以写成：
$$y = w(sx') + b = (ws)x' + b$$

也就是说，只要令
$$w' = ws,\quad b' = b$$
就有
$$y = w'x' + b$$

结论：**你改变了输入坐标轴的单位（把 $x$ 换成 $x'$），对应地权重 $w$ 的数值也会“换单位”变成 $w'$**。模型能表示的函数集合没有变，只是“哪个数值叫做好参数”变了。

**为什么说是“参数空间换坐标系”？**  
参数空间就是所有参数 $(w,b)$ 组成的坐标平面。你做了特征缩放后，最优点的位置会从 $(w^*, b^*)$ 变到 $(w'^*, b^*)$；这像是在同一个几何对象上换了一把尺子（坐标刻度变了）。  
训练时感觉“更稳定/更好收敛”，通常就是因为这个换刻度让 loss 曲面的形状更“圆”（各方向梯度尺度更接近），梯度下降更好走。


### 8. 归一化与标准化

严格来说，<strong>归一化(Normalization)</strong>指的是把数据的取值范围缩放到一个固定区间内，比如 $[-1, 1]$ 或 $[0, 1]$。  
<strong>标准化(Standardization)</strong>指的是对数据减去均值，再除以标准差：

$$x'=\frac{x-\mu}{\sigma}$$

但在机器学习里，由于历史原因，很多资料会用 “Normalization” 这个词来泛指“对特征做尺度变换”。所以如果有人说他对数据进行了“归一化”，但实际代码里做的是“标准化”，你也不用感到奇怪。

---

##### 什么时候用归一化

因为对 feature 进行归一化/标准化通常会让训练更稳定，而且一般不会带来坏处。基本上大多数深度学习模型都会默认对 feature 做归一化（广义的 normalization）操作。